## Image Extractor

Run img_setup.sh for the conda env and other dependencies to run this notebook.

This file takes the PDFs downloaded via scraper.ipynb, converts them into a JPG format, and then identifies/extracts images from the larger article body for CLIP embedding downstream.

While currently set up for CPU, it is quicker to run this in a Google Colab to take advantage of an A100 session. This can be further optimized by switching over to batch inference, but is currently serialized.

In [38]:
# Imports
from pathlib import Path
import os
import json
import cv2
import csv
import detectron2
from tqdm.notebook import tqdm
from tqdm import tqdm
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor
from pdf2image import convert_from_path
from pdf2image.exceptions import PDFPageCountError

In [21]:
# Set paths

BASE_DIR = Path("output")
PDF_DIR = BASE_DIR / "pdfs"
JPG_DIR = BASE_DIR / "page_jpgs"
IMG_DIR = BASE_DIR / "images"

# ensure output dirs exist
JPG_DIR.mkdir(parents=True, exist_ok=True)
IMG_DIR.mkdir(parents=True, exist_ok=True)

Convert PDFs to JPGs by page

In [ ]:
CHECKPOINT_PATH = BASE_DIR / "pdf_to_page.json"

# Load existing checkpoint
if CHECKPOINT_PATH.exists():
    checkpoint = json.loads(CHECKPOINT_PATH.read_text())
else:
    checkpoint = {"completed": [], "failed": []}

for pdf_path in PDF_DIR.rglob("*.pdf"):
    key = str(pdf_path)

    if key in checkpoint["completed"]:
        continue

    rel_path = pdf_path.relative_to(PDF_DIR).parent  # year/month

    out_dir = JPG_DIR / rel_path
    out_dir.mkdir(parents=True, exist_ok=True)

    try:
        pages = convert_from_path(pdf_path)
    except PDFPageCountError as e:
        print(f"Skipping corrupt PDF: {pdf_path}\n  {e}")
        checkpoint["failed"].append(key)
        CHECKPOINT_PATH.write_text(json.dumps(checkpoint, indent=2))
        continue

    base_name = pdf_path.stem  # e.g. mvol-0004-1903-0505

    for i, page in enumerate(pages, start=1):
        out_path = out_dir / f"{base_name}_p{str(i).zfill(2)}.jpg"
        page.save(out_path, "JPEG")

    checkpoint["completed"].append(key)
    CHECKPOINT_PATH.write_text(json.dumps(checkpoint, indent=2))

Setup Newspaper Navigator

In [ ]:
LABEL_MAP = {
    0: "Photograph",
    1: "Illustration",
    2: "Map",
    3: "Comics/Cartoon",
    4: "Editorial Cartoon",
    5: "Headline",
    6: "Advertisement"
}

config_path = os.path.join(
    os.path.dirname(detectron2.__file__),
    "model_zoo", "configs", "COCO-Detection", "faster_rcnn_R_50_FPN_3x.yaml"
)

cfg = get_cfg()
cfg.merge_from_file(config_path)
cfg.MODEL.WEIGHTS = os.path.expanduser("~/newspaper_navigator_model/model_final.pth")
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 7
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
cfg.MODEL.DEVICE = "cpu" # Change if using GPU in Colab

predictor = DefaultPredictor(cfg)

Iteratively extract and store images and image metadata

In [ ]:
CHECKPOINT_PATH = BASE_DIR / "page_to_img.json"
CSV_PATH = BASE_DIR / "img_metadata.csv"

if CHECKPOINT_PATH.exists():
    checkpoint = json.loads(CHECKPOINT_PATH.read_text())
else:
    checkpoint = {"completed": [], "failed": []}

# Write CSV header if new file
if not CSV_PATH.exists():
    with open(CSV_PATH, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["jpg_path", "base_name", "label", "score", "x1", "y1", "x2", "y2", "crop_h", "crop_w", "out_path"])

TARGET_TYPES = {
    "Photograph",
    "Illustration",
    "Map",
    "Comics/Cartoon",
    "Editorial Cartoon"
}

jpg_paths = sorted(JPG_DIR.rglob("*.jpg"))

for jpg_path in tqdm(jpg_paths, desc="Extracting images"):
    key = str(jpg_path)

    if key in checkpoint["completed"]:
        continue

    try:
        rel_path = jpg_path.relative_to(JPG_DIR).parent
        out_dir = IMG_DIR / rel_path
        out_dir.mkdir(parents=True, exist_ok=True)

        image = cv2.imread(str(jpg_path))
        if image is None:
            raise ValueError(f"cv2 could not read image: {jpg_path}")

        outputs = predictor(image)
        instances = outputs["instances"].to("cpu")

        # build detection log for checkpoint
        detection_log = []
        print(f"\n{jpg_path.name} — shape: {image.shape}, detections: {len(instances)}")
        for i in range(len(instances)):
            label = LABEL_MAP[instances.pred_classes[i].item()]
            score = instances.scores[i].item()
            print(f"  {label}: {score:.2f}")
            detection_log.append({"label": label, "score": round(score, 4)})

        figures = []
        for i in range(len(instances)):
            label = LABEL_MAP[instances.pred_classes[i].item()]
            if label not in TARGET_TYPES:
                continue
            box = instances.pred_boxes[i].tensor.numpy()[0]
            score = instances.scores[i].item()
            figures.append((label, score, box))

        figures = sorted(figures, key=lambda f: (f[2][1], f[2][0]))

        base_name = jpg_path.stem

        with open(CSV_PATH, "a", newline="") as f:
            writer = csv.writer(f)
            for i, (label, score, box) in enumerate(figures, start=1):
                x1, y1, x2, y2 = map(int, box)
                crop = image[y1:y2, x1:x2]
                crop_h, crop_w = crop.shape[:2]
                out_path = out_dir / f"{base_name}_{str(i).zfill(2)}.jpg"
                cv2.imwrite(str(out_path), crop)
                writer.writerow([str(jpg_path), base_name, label, f"{score:.4f}", x1, y1, x2, y2, crop_h, crop_w, str(out_path)])

        checkpoint["completed"].append({
            "path": key,
            "detections": detection_log
        })

    except Exception as e:
        print(f"\nFailed: {jpg_path}\n  {e}")
        checkpoint["failed"].append({"path": key, "error": str(e)})

    finally:
        CHECKPOINT_PATH.write_text(json.dumps(checkpoint, indent=2))

#### Once the images are FULLY processed: You can safely delete page_jpgs/ to save storage

In [ ]:
# Delete page-level JPGs; requires manual confirmation

confirm = input(f"Type 'yes' to delete {JPG_DIR} and all its contents: ")

if confirm.strip().lower() == "yes":
    import shutil
    shutil.rmtree(JPG_DIR)
    print(f"Deleted {JPG_DIR}")
else:
    print("Aborted.")